In [2]:
import os
import shutil
import cv2
import pickle
import random

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import load_model

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

After noticing that the pictures with the *.jpeg extension are the ones with the butterfly, we proceed to split the dataset in 2 folders where we have all the imgs with and without the butterflies so we can tag them and from there train out  our CNN

In [3]:
src1 = r'X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\test'
src2 = r'X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\train'
dst1 = r'X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\with'
dst2 = r'X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\without'

In [4]:
img_counter = 1

for src_folder in [src1, src2]:
    
    for img in os.listdir(src_folder):
        
        src_path = os.path.join(src_folder, img)
        
        if img.endswith(".jpeg"):
            new_name=f'label_1_img{img_counter}.jpeg'
    
            src_path = os.path.join(src_folder, img)
            dest_path = os.path.join(dst1, new_name)
    
            print(f'Butterfly copied: {img} -> {new_name} st {dst1}')
    
        else:
            ext = os.path.splitext(img)[1]
            new_name=f'label_1_img{img_counter}{ext}'
    
            src_path = os.path.join(src_folder, img)
            dest_path = os.path.join(dst2, new_name)
    
            print(f'NO Butterfly copied: {img} -> {new_name} at {dst2}')
        
        shutil.copy2(src_path,dest_path)    
        img_counter +=1

NO Butterfly copied: imagen_1.jpg -> label_1_img1.jpg at X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\without
NO Butterfly copied: imagen_10.jpg -> label_1_img2.jpg at X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\without
Butterfly copied: imagen_100.jpeg -> label_1_img3.jpeg st X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\with
NO Butterfly copied: imagen_101.jpg -> label_1_img4.jpg at X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\without
NO Butterfly copied: imagen_102.jpg -> label_1_img5.jpg at X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\without
NO Butterfly copied: imagen_103.jpg -> label_1_img6.jpg at X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\without
Butterfly copied: imagen_104.jpeg -> label_1_img7.jpeg st X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\with
NO Butterfly copied: imagen_105.jpg -> label_1_img8.jpg at X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\without
Butterfly copied: imagen_106.jpeg -> label_1_img9.jpeg st X:\CODING\PROJECTS\BUTTERFLY\2nd_Try\with
Butterfly copied: imagen_107.jpeg -> label_1_img10.jpeg st X:\CODING\PROJECTS\B

Now we have our dataset separated in "with" and "without". A preprocess of the data we will continue right below

In [8]:
count = 0
for img in os.listdir(dst1):
    count +=1
print(f"Imgs WITH a butterfly: {count}")

print()

count = 0
for img in os.listdir(dst2):
    count +=1
print(f"Imgs WITHOUT a butterfly: {count}")

Imgs WITH a butterfly: 1487

Imgs WITHOUT a butterfly: 2725


In [4]:
without_imgs = os.listdir(dst2)
selected_without = random.sample(without_imgs, 1487)

In [5]:
balanced_files = []
balanced_labels = []

for img in os.listdir(dst1):
    balanced_files.append(os.path.join(dst1, img))
    balanced_labels.append(1) # 1 Will be the label for those imgs where THERE IS a butterfly

for img in os.listdir(dst2):
    balanced_files.append(os.path.join(dst2, img))
    balanced_labels.append(0) # 0 Will be the label for those imgs where there is NO butterfly
    
combined = list(zip(balanced_files, balanced_labels))
random.shuffle(combined)
balanced_files[:], balanced_labels[:] = zip(*combined)

In [11]:
def load_imgs (file_paths, target_size = (250,250)):
    imgs = []
    for path in file_paths:
        img = image.load_img(path, target_size=target_size)
        img_array = image.img_to_array(img)/255.0
        imgs.append(img_array)
    return np.array(imgs)
X = load_imgs(balanced_files)
y = np.array(balanced_labels)

In [14]:
X_train, X_val, y_train, y_val = train_test_split(X, y,test_size=0.2,random_state=42)

In [15]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(250, 250, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary: with/without butterfly
])

In [16]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'Precision', 'Recall']
)

In [17]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=30
)

Epoch 1/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - Precision: 0.6776 - Recall: 0.5113 - accuracy: 0.7322 - loss: 0.8101 - val_Precision: 0.9927 - val_Recall: 0.9410 - val_accuracy: 0.9775 - val_loss: 0.0639
Epoch 2/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 178s 2s/step - Precision: 0.9841 - Recall: 0.9810 - accuracy: 0.9875 - loss: 0.0547 - val_Precision: 0.9925 - val_Recall: 0.9201 - val_accuracy: 0.9703 - val_loss: 0.0835
Epoch 3/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 175s 2s/step - Precision: 0.9662 - Recall: 0.9776 - accuracy: 0.9800 - loss: 0.0559 - val_Precision: 0.9930 - val_Recall: 0.9896 - val_accuracy: 0.9941 - val_loss: 0.0241
Epoch 4/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 174s 2s/step - Precision: 0.9930 - Recall: 1.0000 - accuracy: 0.9975 - loss: 0.0113 - val_Precision: 0.9863 - val_Recall: 1.0000 - val_accuracy: 0.9953 - val_loss: 0.0244
Epoch 5/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 174s 2s/step - Precision: 0.9970 - Recall: 0.9948 - accuracy: 0.9970 - loss: 0.0113 - val_Precision: 0.9965 - val_Re

In [18]:
# model.save('butterfly_classifier.h5')

In [19]:
def preprocess_image(img_path, target_size=(250, 250)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img) / 255.0  # Normalizar
    img_array = np.expand_dims(img_array, axis=0)  # Añadir dimensión del batch (1, 150, 150, 3)
    return img_array

In [50]:
# img with butterfly

indices_con_mariposa = [i for i, label in enumerate(y_val) if label == 1]

img_test_1 = X_val[indices_con_mariposa[175]]
label_test_1 = y_val[indices_con_mariposa[175]]

In [51]:
# img AND label (no butterfly) 

indices_con_mariposa2 = [i for i, label in enumerate(y_val) if label == 0]

img_test_0 = X_val[indices_con_mariposa[175]]
label_test_0 = y_val[indices_con_mariposa[175]]

In [53]:
img_test_0.shape

(250, 250, 3)

In [54]:
rd_img = np.expand_dims(img_test_0, axis=0)

In [56]:
prediction = model.predict(rd_img)[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step


In [62]:
prediction

0.99999964

In [3]:
model = load_model('butterfly_classifier.h5')  

In [17]:
def preprocess_image(img_path, target_size=(250, 250)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img) / 255.0 
    img_array = np.expand_dims(img_array, axis=0)  
    return img_array

In [22]:
img_path = r'X:\CODING\PROJECTS\BUTTERFLY\test\imagen_11.jpg'
processed_img = preprocess_image(img_path)

In [23]:
prediction = model.predict(processed_img)

if prediction[0][0] > 0.5:
    print("✅ There IS a butterfly in the picture (prob: {:.2f}%)".format(prediction[0][0] * 100))
else:
    print("❌ There is NO butterfly in the picture (prob: {:.2f}%)".format((1 - prediction[0][0]) * 100))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
❌ There is NO butterfly in the picture (prob: 100.00%)
